In [39]:
import pandas as pd
import gc
import os
import glob

In [41]:
folder_path = r"C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo"
output_file = r"C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Tratado_antigo\\"
files = glob.glob(os.path.join(folder_path, "**", "planilha_*.csv"), recursive=True)
files = [f for f in files if "completa" not in f and "agrupada" not in f]
files.sort()

value_vars = [
    "numero_de_operacoes",
    "a_vencer_ate_90_dias",
    "a_vencer_de_91_ate_360_dias",
    "a_vencer_de_361_ate_1080_dias",
    "a_vencer_de_1081_ate_1800_dias",
    "a_vencer_de_1801_ate_5400_dias",
    "a_vencer_acima_de_5400_dias",
    "vencido_acima_de_15_dias",
    "carteira_ativa",
    "carteira_inadimplida_arrastada",
    "ativo_problematico"
]

id_vars = [
    "data_base", "uf", "tcb", "sr", "cliente", "ocupacao", 
    "cnae_secao", "cnae_subclasse", "porte", "modalidade", "origem", "indexador"
]


In [ ]:
for file_idx, filepath in enumerate(files):
    print(file_idx, filepath )
    chunk = pd.read_csv(filepath, sep=";",encoding="utf-8-sig", dtype=str,keep_default_na=False)
    #chunk.rename(columns=lambda x: x.replace('ï»¿', '').replace('\ufeff', '').strip(), inplace=True)
    for col in id_vars:
            if col in chunk.columns:
                chunk[col] = chunk[col].str.strip()
    if 'numero_de_operacoes' in chunk.columns:
            chunk['numero_de_operacoes'] = chunk['numero_de_operacoes'].astype(str).str.replace('<= 15', '7').str.replace('<=15', '7').str.strip()
            chunk['numero_de_operacoes'] = pd.to_numeric(chunk['numero_de_operacoes'], errors='coerce').fillna(0).astype('float64')
    cols_to_melt = [c for c in value_vars if c in chunk.columns]
    chunk[cols_to_melt] = (
    chunk[cols_to_melt]
    .replace(",", ".", regex=True)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype("float64")
    )
    melted = chunk.melt(id_vars=id_vars, value_vars=cols_to_melt, var_name='Atributo', value_name='Valor')
        
    grouped = melted.groupby(id_vars + ['Atributo'], as_index=False,dropna=False)['Valor'].sum()
    # Valor numérico
    grouped['Valor'] = pd.to_numeric(grouped['Valor'], errors='coerce').fillna(0)
    

    # data_base como data
    grouped['data_base'] = pd.to_datetime(grouped['data_base'], errors='coerce')

    # todas as outras colunas como string
    cols_str = [c for c in grouped.columns if c not in ['Valor', 'data_base']]
    grouped[cols_str] = grouped[cols_str].astype(str)
    grouped = grouped[grouped['Valor'] != 0]
    grouped.to_parquet(output_file+str(os.path.basename(filepath))+'.parquet')  
    #grouped = grouped[grouped['Valor'] != 0]

    #chunk.to_csv(output_file+str(os.path.basename(filepath))+'.csv',  sep=';',decimal=',' ,encoding="utf-8-sig", index=False)  
    #grouped.to_csv(output_file+str(file_idx)+'grouped.csv',  sep=';',decimal=',' ,encoding="utf-8-sig", index=False) 
     

0 C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo\planilha_2012\planilha_201201.csv
1 C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo\planilha_2012\planilha_201202.csv
2 C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo\planilha_2012\planilha_201203.csv
3 C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo\planilha_2012\planilha_201204.csv
4 C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo\planilha_2012\planilha_201205.csv
5 C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo\planilha_2012\planilha_201206.csv
6 C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo\planilha_2012\planilha_201207.csv
7 C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo\planilha_2012\planilha_201208.csv
8 C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo\planilha_2012\planilha_201209.csv
9 C:\Users\samue\OneDrive\Documentos\Repositorios\BACEN\Bases\Antigo\plan